# DermaLens AI | ResNet-50 training (Kaggle)
**Research workflow:** frozen-backbone head training -> partial layer4 fine-tuning -> optional full fine-tuning.
Supports `real`, `synthetic`, and `combined` training modes. **No new splitting**: consumes the previously frozen train/val/test folders.

**Pretrained weights required for real runs.** A random-weight switch exists only for local engineering smoke tests.

Images are inspected for errors/near-empty black or white content before training; evaluation datasets with damaged images trigger an error rather than silently changing the test split.
Training uses aspect-preserving 224×224 padding (not center cropping, so facial lesions near edges are not cut off), mild train-only augmentation, and ImageNet normalization.
Official ResNet-50 V2 center-crop preprocessing differs; we explicitly adopt letterboxing for complete facial field-of-view. Review previews to confirm that this is suitable for your collection.

**Run instructions:** Attach your *saved splitting notebook outputs* as Kaggle Input datasets (either a folder containing `train/val/test` or its `.zip`). Edit **only the configuration cell**. If offline, attach official ImageNet V2 ResNet50 `resnet50-11ad3fa6.pth` weights and set `PRETRAINED_WEIGHTS_PATH`.

Legacy notebook evidence is saved under `OUTPUT_ROOT`: `notebook_console.log`, `run_log.jsonl`, model checkpoints and existing report CSVs. For paper comparisons, prefer the newer E0–E5 manifest-driven trainers; this legacy notebook has its own training regime.


## 1. Configuration — edit the paths and training modes

In [ ]:
from pathlib import Path

# Folder root containing train/val/test OR a .zip from the previous real splitting notebook.
REAL_SOURCE = "/kaggle/input/EDIT_REAL_SPLIT_DATASET/facial_real_split.zip"
# Optional for real-only training, required for synthetic/combined training and synthetic testing.
SYN_SOURCE = ""  # Set when ready for synthetic test or synthetic/combined training.

RUN_MODES = ["real"]  # Choose ["real", "synthetic", "combined"] to train all three sequentially.
PRETRAINED_WEIGHTS_PATH = ""  # If empty, torchvision downloads/caches ImageNet V2 weights (requires Internet for first use).
OUTPUT_ROOT = "/kaggle/working/dermalens_resnet50"

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 24  # Reduce to 12 or 8 if Kaggle GPU memory is insufficient.
NUM_WORKERS = 2
EPOCHS_HEAD = 6
EPOCHS_PARTIAL = 15
EPOCHS_FULL = 0  # Optional; set >0 only if validation justifies full fine-tuning.
EARLY_STOPPING_PATIENCE = 5
LR_HEAD = 1e-3
LR_PARTIAL_HEAD = 1e-4
LR_PARTIAL_LAYER4 = 2e-5
LR_FULL_HEAD = 5e-5
LR_FULL_BACKBONE = 5e-6
WEIGHT_DECAY = 1e-4
USE_AMP = True
LOSS_CLASS_WEIGHTING = "none"  # "none" or "sqrt_inverse"; do not add weighted sampling for single domain at the same time.
COMBINED_DOMAIN_CLASS_BALANCED_SAMPLING = True
STRICT_EVAL_IMAGE_VALIDATION = True
RUN_PREPROCESSING_PREVIEW = True
PACKAGE_RESULTS_AS_ZIP = True
SAVE_HEAD_ONLY_BASELINE = True  # Preserve linear-probe/head-only result separately from partial fine-tuning.

# Engineering test only; must remain False for research runs.
ALLOW_RANDOM_WEIGHTS_FOR_SMOKE_TEST = False

CLASS_NAMES = ["Acne", "Fungal_Infection", "Vitiligo", "Hyperpigmentation", "Skin_Cancer", "Normal"]

In [ ]:
# Legacy ResNet50 trainer: retain failures, progress and checkpoints in Kaggle outputs.
import subprocess,sys
from pathlib import Path
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1',
                           'https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log
PRIVATE_CONSOLE=start_notebook_logging(OUTPUT_ROOT,kind='legacy_ResNet50_training',
    config={'training_modes':RUN_MODES,'seed':SEED},repo_root=DERMSYNREAL_REPO)
print('Legacy research log:',PRIVATE_CONSOLE)


## 2. Imports, reproducibility, and source discovery

In [ ]:
import csv
import hashlib
import json
import math
import os
import random
import shutil
import sys
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps, UnidentifiedImageError
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from torchvision.transforms import InterpolationMode

assert len(CLASS_NAMES) == len(set(CLASS_NAMES)) == 6
assert RUN_MODES and set(RUN_MODES).issubset({"real", "synthetic", "combined"})
assert EPOCHS_HEAD > 0 and EPOCHS_PARTIAL >= 0 and EPOCHS_FULL >= 0
assert IMAGE_SIZE >= 128 and BATCH_SIZE > 0
assert LOSS_CLASS_WEIGHTING in {"none", "sqrt_inverse"}
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, max(1, os.cpu_count() or 1)))
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = False

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(SEED)
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, "torchvision:", __import__("torchvision").__version__)
print("Pillow:", PIL.__version__, "sklearn:", sklearn.__version__, "device:", DEVICE)
print("Training modes:", RUN_MODES)
print("Note: T4 GPU is recommended for full dataset training; CPU is suitable for a short smoke test.")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

def find_split_root(folder):
    folder = Path(folder)
    def has_splits(p):
        return all((p / split).is_dir() for split in ("train", "val", "test"))
    if has_splits(folder):
        return folder
    candidates = []
    for base, dirs, _ in os.walk(folder):
        current = Path(base)
        depth = len(current.relative_to(folder).parts)
        if depth > 3:
            dirs[:] = []
            continue
        if has_splits(current):
            candidates.append(current)
            dirs[:] = []
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one train/val/test root under {folder}; found {len(candidates)}: {candidates[:5]}")
    return candidates[0]

def resolve_source(source, domain):
    if not source:
        return None
    p = Path(source).expanduser()
    if not p.exists():
        raise FileNotFoundError(f"{domain} source not found: {p}. Edit your source path in the first cell.")
    if p.is_dir():
        return find_split_root(p)
    if not (p.is_file() and p.suffix.lower() == ".zip"):
        raise ValueError(f"{domain}: expected a directory or .zip containing train/val/test, got: {p}")
    extracted = OUTPUT_ROOT / f"_extracted_{domain}"
    ready = extracted / "_complete.json"
    stat = p.stat()
    signature = {"source": str(p.resolve()), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
    if ready.is_file():
        try:
            if json.loads(ready.read_text()) == signature:
                return find_split_root(extracted)
        except (json.JSONDecodeError, ValueError):
            pass
    if extracted.exists():
        shutil.rmtree(extracted)
    extracted.mkdir(parents=True)
    with zipfile.ZipFile(p) as z:
        bad_member = z.testzip()
        if bad_member:
            raise ValueError(f"Corrupted ZIP member: {bad_member}")
        target_dir = extracted.resolve()
        for m in z.infolist():
            dest = (extracted / m.filename).resolve()
            if not dest.is_relative_to(target_dir):
                raise ValueError(f"Unsafe archive path: {m.filename}")
        z.extractall(extracted)
    root = find_split_root(extracted)
    ready.write_text(json.dumps(signature))
    return root

## 3. Validate every image and audit exact cross-split leakage
Validation does not change your upstream split. Invalid **train** images may be excluded with a report.
Invalid **validation/test** images trigger an error by default: fix/re-export the frozen split rather than silently changing its evaluation population.

In [ ]:
def load_rgb(path):
    # EXIF correction before anything else; do not convert transparent pixels to black.
    with Image.open(path) as original:
        original.load()  # detect truncated/corrupt images early
        oriented = ImageOps.exif_transpose(original)
        if oriented.mode == "RGB":
            rgb = oriented.copy()
        elif "A" in oriented.getbands() or (oriented.mode == "P" and "transparency" in oriented.info):
            rgba = oriented.convert("RGBA")
            rgb = Image.new("RGB", rgba.size, (255, 255, 255))
            rgb.paste(rgba, mask=rgba.getchannel("A"))
        else:
            rgb = oriented.convert("RGB")
        return rgb

def hash_bytes(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def inspect_image(path):
    try:
        rgb = load_rgb(path)
        w, h = rgb.size
        if min(w, h) < 64:
            return {"valid": False, "reason": f"too_small_{w}x{h}"}
        # A conservative near-blank audit, *not* a skin-tone or clinical-quality classifier.
        small = np.asarray(rgb.resize((64, 64), Image.Resampling.BILINEAR), dtype=np.uint8)
        gray = small.astype(np.float32).mean(axis=2)
        if float((gray < 8).mean()) > 0.995 and float(gray.std()) < 5:
            return {"valid": False, "reason": "almost_all_black"}
        if float((gray > 247).mean()) > 0.995 and float(gray.std()) < 5:
            return {"valid": False, "reason": "almost_all_white"}
        pixel = hashlib.sha256()
        pixel.update(f"{w}x{h}:RGB:".encode())
        pixel.update(np.asarray(rgb).tobytes())
        return {"valid": True, "width": w, "height": h, "file_sha256": hash_bytes(path), "pixel_sha256": pixel.hexdigest()}
    except (OSError, ValueError, UnidentifiedImageError, Image.DecompressionBombError) as exc:
        return {"valid": False, "reason": f"decode_error: {type(exc).__name__}: {exc}"}

def inventory_domain(split_root, domain):
    rows, bad = [], []
    for split in ("train", "val", "test"):
        for cls in CLASS_NAMES:
            directory = split_root / split / cls
            if not directory.is_dir():
                raise FileNotFoundError(f"Required directory is missing: {directory}")
            files = sorted(p for p in directory.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
            for path in files:
                rec = {"domain": domain, "split": split, "class_name": cls, "label": CLASS_TO_INDEX[cls], "path": str(path)}
                info = inspect_image(path)
                if info["valid"]:
                    rows.append({**rec, **info})
                else:
                    bad.append({**rec, "reason": info["reason"]})
    return rows, bad

def fail_on_leakage(rows):
    # Detect byte-exact and canonical-pixel-exact overlap across train/val/test,
    # including cross-domain contamination. Visual duplicates are handled upstream.
    problems = []
    for hash_col in ("file_sha256", "pixel_sha256"):
        by_hash = defaultdict(list)
        for row in rows:
            by_hash[row[hash_col]].append(row)
        for digest, shared in by_hash.items():
            splits = set(r["split"] for r in shared)
            classes = set(r["class_name"] for r in shared)
            if len(splits) > 1 or len(classes) > 1:
                problems.append({"hash_type": hash_col, "hash": digest, "splits": sorted(splits),
                                 "classes": sorted(classes), "paths": [r["path"] for r in shared]})
    if problems:
        dest = OUTPUT_ROOT / "cross_split_or_label_conflicts.json"
        dest.write_text(json.dumps(problems, indent=2))
        raise RuntimeError(f"Found {len(problems)} exact hash split/label conflicts. See {dest}. Correct upstream splits before training.")
    return True

need_synthetic = "synthetic" in RUN_MODES or "combined" in RUN_MODES
need_real = "real" in RUN_MODES or "combined" in RUN_MODES
if need_real and not REAL_SOURCE:
    raise ValueError("A real source is required by RUN_MODES.")
if need_synthetic and not SYN_SOURCE:
    raise ValueError("A synthetic source is required by RUN_MODES.")
roots = {}
if REAL_SOURCE:
    roots["real"] = resolve_source(REAL_SOURCE, "real")
if SYN_SOURCE:
    roots["synthetic"] = resolve_source(SYN_SOURCE, "synthetic")

ROWS, BAD_ROWS = [], []
for domain, root in roots.items():
    good, bad = inventory_domain(root, domain)
    ROWS.extend(good)
    BAD_ROWS.extend(bad)
    print(f"{domain}: {len(good)} valid images, {len(bad)} invalid images")
pd.DataFrame(BAD_ROWS, columns=["domain", "split", "class_name", "label", "path", "reason"]).to_csv(OUTPUT_ROOT / "invalid_images.csv", index=False)
if STRICT_EVAL_IMAGE_VALIDATION and any(r["split"] in {"val", "test"} for r in BAD_ROWS):
    raise RuntimeError("Invalid validation/test images found. See invalid_images.csv. Fix your frozen split, then restart.")
fail_on_leakage(ROWS)
for domain in roots:
    for split in ("train", "val", "test"):
        counts = Counter(r["class_name"] for r in ROWS if r["domain"] == domain and r["split"] == split)
        if any(counts[c] == 0 for c in CLASS_NAMES):
            raise ValueError(f"{domain}/{split}: one or more of the six classes has no valid images: {dict(counts)}")
        print(domain, split, dict(counts))
pd.DataFrame(ROWS).to_csv(OUTPUT_ROOT / "validated_image_inventory.csv", index=False)
print("Preflight passed: nonblank decodable images in every class; no exact-hash overlap across partitions.")

## 4. Image preprocessing: full-face letterbox, mild augmentation, preflight preview
Displayed examples are **denormalized** before plotting, so normalized dark values are not mistaken for black images.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
PAD_RGB = (124, 116, 104)  # neutral border close to normalized zero; avoids black letterbox bars

class Letterbox:
    """Resize without cropping or distortion, then center-pad to IMAGE_SIZE."""
    def __init__(self, size=224, fill=PAD_RGB):
        self.size = size
        self.fill = fill

    def __call__(self, im):
        fit = ImageOps.contain(im, (self.size, self.size), method=Image.Resampling.BICUBIC)
        result = Image.new("RGB", (self.size, self.size), self.fill)
        result.paste(fit, ((self.size - fit.width) // 2, (self.size - fit.height) // 2))
        return result

train_transform = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7, interpolation=InterpolationMode.BILINEAR, fill=PAD_RGB),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FacialDataset(Dataset):
    def __init__(self, records, transform):
        self.records = list(records)
        self.transform = transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        record = self.records[idx]
        img = load_rgb(record["path"])
        tensor = self.transform(img)
        if tensor.shape != (3, IMAGE_SIZE, IMAGE_SIZE) or not torch.isfinite(tensor).all():
            raise RuntimeError(f"Preprocessing failed for {record['path']}")
        return tensor, record["label"], record["path"]

def to_display(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

if RUN_PREPROCESSING_PREVIEW:
    fig, axes = plt.subplots(2, 6, figsize=(17, 6))
    for cidx, cls in enumerate(CLASS_NAMES):
        for ridx, domain in enumerate(("real", "synthetic")):
            ax = axes[ridx, cidx]
            options = [r for r in ROWS if r["domain"] == domain and r["split"] == "train" and r["class_name"] == cls]
            if options:
                tensor = eval_transform(load_rgb(options[0]["path"]))
                rendered = to_display(tensor)
                if rendered.mean() < 0.025 or rendered.std() < 0.007:
                    raise RuntimeError(f"Unexpected dark/flat preprocessing: {options[0]['path']}")
                ax.imshow(rendered)
            else:
                ax.text(0.5, 0.5, "Not attached", ha="center", va="center")
            ax.set_title(f"{domain}: {cls}", fontsize=8)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / "preprocessing_preview.png", dpi=140)
    plt.show()
    plt.close(fig)
print("Sample preprocessing shape:", eval_transform(load_rgb(ROWS[0]["path"])).shape)

## 5. Dataloaders, pretrained backbone, and progressive fine-tuning
BatchNorm running statistics are frozen throughout (especially important for small clinical datasets/batches).
Real+synthetic mode uses balanced domain×class sampling **only on train**, to avoid synthetic images dominating by sheer count.

In [ ]:
def subset(domain, split):
    return [r for r in ROWS if r["domain"] == domain and r["split"] == split]

def make_loader(records, train=False, mode="real"):
    ds = FacialDataset(records, train_transform if train else eval_transform)
    sampler = None
    shuffle = train
    if train and mode == "combined" and COMBINED_DOMAIN_CLASS_BALANCED_SAMPLING:
        counts = Counter((r["domain"], r["class_name"]) for r in records)
        # Each domain contributes 50%; each of its six classes contributes equally.
        weights = [1.0 / (2 * len(CLASS_NAMES) * counts[(r["domain"], r["class_name"])]) for r in records]
        sampler = WeightedRandomSampler(weights, num_samples=len(records), replacement=True,
                                        generator=torch.Generator().manual_seed(SEED))
        shuffle = False
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                      persistent_workers=(NUM_WORKERS > 0),
                      worker_init_fn=seed_worker if NUM_WORKERS else None)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_pretrained_resnet():
    if ALLOW_RANDOM_WEIGHTS_FOR_SMOKE_TEST:
        print("WARNING: RANDOM weights for engineering tests only; NOT valid for research.")
        network = models.resnet50(weights=None)
        weight_origin = "RANDOM_ENGINEERING_TEST"
    elif PRETRAINED_WEIGHTS_PATH:
        weights_file = Path(PRETRAINED_WEIGHTS_PATH)
        if not weights_file.is_file():
            raise FileNotFoundError(f"Pretrained weights missing: {weights_file}")
        network = models.resnet50(weights=None)
        weights = torch.load(weights_file, map_location="cpu", weights_only=True)
        if isinstance(weights, dict) and "state_dict" in weights:
            weights = weights["state_dict"]
        network.load_state_dict(weights, strict=True)
        weight_origin = str(weights_file)
    else:
        try:
            network = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        except Exception as exc:
            raise RuntimeError("Cannot load ResNet-50 ImageNet V2 weights. Enable Kaggle Internet or attach the "
                               "official resnet50-11ad3fa6.pth weight file and edit PRETRAINED_WEIGHTS_PATH.") from exc
        weight_origin = "torchvision/ResNet50_Weights.IMAGENET1K_V2"
    in_features = network.fc.in_features
    network.fc = nn.Linear(in_features, len(CLASS_NAMES))
    return network.to(DEVICE), weight_origin

def freeze_batchnorm(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            for param in module.parameters():
                param.requires_grad_(False)

def set_training_stage(model, stage):
    for p in model.parameters():
        p.requires_grad_(False)
    if stage == "head":
        for p in model.fc.parameters():
            p.requires_grad_(True)
    elif stage == "partial":
        for p in model.layer4.parameters():
            p.requires_grad_(True)
        for p in model.fc.parameters():
            p.requires_grad_(True)
    elif stage == "full":
        for p in model.parameters():
            p.requires_grad_(True)
    else:
        raise ValueError(stage)
    freeze_batchnorm(model)

def make_optimizer(model, stage):
    fc = [p for p in model.fc.parameters() if p.requires_grad]
    if stage == "head":
        groups = [{"params": fc, "lr": LR_HEAD}]
    elif stage == "partial":
        layer4 = [p for p in model.layer4.parameters() if p.requires_grad]
        groups = [{"params": layer4, "lr": LR_PARTIAL_LAYER4}, {"params": fc, "lr": LR_PARTIAL_HEAD}]
    else:
        ids = {id(p) for p in model.fc.parameters()}
        other = [p for p in model.parameters() if p.requires_grad and id(p) not in ids]
        groups = [{"params": other, "lr": LR_FULL_BACKBONE}, {"params": fc, "lr": LR_FULL_HEAD}]
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def make_criterion(train_records):
    if LOSS_CLASS_WEIGHTING == "none":
        return nn.CrossEntropyLoss()
    counts = Counter(r["label"] for r in train_records)
    weights = torch.tensor([1 / math.sqrt(counts[i]) for i in range(len(CLASS_NAMES))], dtype=torch.float32)
    weights /= weights.mean()
    return nn.CrossEntropyLoss(weight=weights.to(DEVICE))

def inference(model, loader, criterion=None):
    model.eval()
    labels, preds, probs, names = [], [], [], []
    losses = []
    with torch.inference_mode():
        for images, targets, paths in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(USE_AMP and DEVICE.type == "cuda")):
                logits = model(images)
                if criterion is not None:
                    losses.append((float(criterion(logits, targets).item()), len(targets)))
            prob = logits.float().softmax(dim=1).cpu().numpy()
            labels.extend(targets.cpu().tolist())
            preds.extend(prob.argmax(axis=1).tolist())
            probs.extend(prob.tolist())
            names.extend(paths)
    loss_value = sum(v * n for v, n in losses) / max(1, sum(n for _, n in losses)) if losses else None
    metrics = {"accuracy": float(accuracy_score(labels, preds)),
               "macro_f1": float(f1_score(labels, preds, labels=list(range(6)), average="macro", zero_division=0)),
               "balanced_accuracy": float(balanced_accuracy_score(labels, preds)), "loss": loss_value, "n": len(labels)}
    return metrics, labels, preds, probs, names

def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    freeze_batchnorm(model)  # model.train() would otherwise update running statistics.
    total, total_n = 0.0, 0
    for images, targets, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(USE_AMP and DEVICE.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss. Inspect preprocessing, images, and learning rate.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total += float(loss.item()) * len(targets)
        total_n += len(targets)
    return total / total_n

def is_better(metrics, best):
    if best is None:
        return True
    return (metrics["macro_f1"] > best["macro_f1"] + 1e-6 or
            (abs(metrics["macro_f1"] - best["macro_f1"]) <= 1e-6 and metrics["loss"] < best["loss"] - 1e-8))

## 6. Train and save **best-validation** checkpoint per mode; reload it for evaluation
No test set is used for tuning, early stopping, checkpoint selection, or sampling.
Synthetic-only selects on synthetic val. Combined selects on real val (held out from training).

In [ ]:
def save_model(path, model, mode, stage, weight_origin, score, history, train_rows):
    payload = {
        "arch": "torchvision.resnet50", "classes": CLASS_NAMES, "class_to_index": CLASS_TO_INDEX,
        "stage": stage, "mode": mode, "best_val": score,
        "pretrained_origin": weight_origin,
        "research_valid": not ALLOW_RANDOM_WEIGHTS_FOR_SMOKE_TEST,
        "image_size": IMAGE_SIZE, "image_preprocessing": "PIL_EXIF_RGB_letterbox_224_ImageNet_mean_std",
        "normalization_mean": IMAGENET_MEAN, "normalization_std": IMAGENET_STD,
        "training_seed": SEED, "training_record_count": len(train_rows),
        "config": {"epochs_head": EPOCHS_HEAD, "epochs_partial": EPOCHS_PARTIAL, "epochs_full": EPOCHS_FULL,
                   "batch_size": BATCH_SIZE, "class_weighting": LOSS_CLASS_WEIGHTING,
                   "balanced_combined_sampling": COMBINED_DOMAIN_CLASS_BALANCED_SAMPLING},
        "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
    }
    torch.save(payload, path)

def reload_best(path):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    assert payload["classes"] == CLASS_NAMES and payload["arch"] == "torchvision.resnet50"
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    model.load_state_dict(payload["state_dict"], strict=True)
    model.to(DEVICE).eval()
    return model, payload

def train_mode(mode):
    seed_all(SEED)
    print("\n" + "=" * 70 + f"\nSTART MODE {mode.upper()}\n" + "=" * 70)
    run_dir = OUTPUT_ROOT / f"resnet50_{mode}"
    run_dir.mkdir(parents=True, exist_ok=True)
    if mode == "real":
        training_records, validation_records = subset("real", "train"), subset("real", "val")
    elif mode == "synthetic":
        training_records, validation_records = subset("synthetic", "train"), subset("synthetic", "val")
    else:
        training_records = subset("real", "train") + subset("synthetic", "train")
        validation_records = subset("real", "val")
    train_loader = make_loader(training_records, train=True, mode=mode)
    val_loader = make_loader(validation_records)
    model, weight_origin = make_pretrained_resnet()
    criterion = make_criterion(training_records)
    history, best = [], None
    checkpoint = run_dir / f"resnet50_{mode}_best.pth"
    stages = [("head", EPOCHS_HEAD), ("partial", EPOCHS_PARTIAL), ("full", EPOCHS_FULL)]
    for stage, budget in stages:
        if budget <= 0:
            continue
        if best is not None:
            model, _ = reload_best(checkpoint)
        set_training_stage(model, stage)
        optimizer = make_optimizer(model, stage)
        scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))
        # Cosine decay over the stage; use a minimum 10% of initial learning rates.
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, budget), eta_min=0)
        stage_best, stale = None, 0
        for epoch in range(1, budget + 1):
            t0 = time.perf_counter()
            training_loss = train_epoch(model, train_loader, criterion, optimizer, scaler)
            val_metrics, *_ = inference(model, val_loader, criterion)
            row = {"mode": mode, "stage": stage, "epoch": epoch, "train_loss": training_loss, **{f"val_{k}":v for k,v in val_metrics.items()},
                   "seconds": round(time.perf_counter() - t0, 2)}
            history.append(row)
            print(f"{mode:9} {stage:7} epoch {epoch:02d}/{budget} loss={training_loss:.4f} "
                  f"val_f1={val_metrics['macro_f1']:.4f} val_acc={val_metrics['accuracy']:.4f}")
            if is_better(val_metrics, stage_best):
                stage_best = val_metrics.copy()
                stale = 0
                if stage == "head" and SAVE_HEAD_ONLY_BASELINE:
                    save_model(run_dir / f"resnet50_{mode}_head_only_best.pth", model, mode,
                               stage, weight_origin, val_metrics, history, training_records)
            else:
                stale += 1
            if is_better(val_metrics, best):
                best = val_metrics.copy()
                save_model(checkpoint, model, mode, stage, weight_origin, best, history, training_records)
                print("  -> saved new best-validation checkpoint")
            scheduler.step()
            if stale >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping: {stage} stage did not improve for {stale} epochs")
                break
        pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
    if not checkpoint.exists():
        raise RuntimeError("Training did not produce a checkpoint.")
    best_model, metadata = reload_best(checkpoint)
    # Independent reload sanity: check logits from the reloaded model are finite.
    sample_loader = make_loader(validation_records[:min(BATCH_SIZE, len(validation_records))])
    sample_batch = next(iter(sample_loader))[0].to(DEVICE)
    with torch.inference_mode():
        logits = best_model(sample_batch)
    if tuple(logits.shape) != (len(sample_batch), 6) or not torch.isfinite(logits).all():
        raise RuntimeError("Reloaded checkpoint did not pass inference sanity test.")
    test_metrics = []
    for test_domain in ("real", "synthetic"):
        if test_domain not in roots:
            continue
        test_loader = make_loader(subset(test_domain, "test"))
        scores, targets, preds, probabilities, paths = inference(best_model, test_loader)
        scores.update({"training_mode": mode, "test_domain": test_domain, "checkpoint_stage": metadata["stage"]})
        test_metrics.append(scores)
        print(f"{mode} -> {test_domain} TEST: macro_F1={scores['macro_f1']:.4f}, accuracy={scores['accuracy']:.4f}")
        preds_frame = pd.DataFrame({"path": paths, "true_label": targets, "predicted_label": preds,
                                    "true_class": [CLASS_NAMES[t] for t in targets],
                                    "predicted_class": [CLASS_NAMES[p] for p in preds]})
        for i, cls in enumerate(CLASS_NAMES):
            preds_frame[f"prob_{cls}"] = [p[i] for p in probabilities]
        preds_frame.to_csv(run_dir / f"test_predictions_{test_domain}.csv", index=False)
        report = classification_report(targets, preds, labels=list(range(6)),
                                       target_names=CLASS_NAMES, zero_division=0, output_dict=True)
        (run_dir / f"classification_report_{test_domain}.json").write_text(json.dumps(report, indent=2))
        cm = confusion_matrix(targets, preds, labels=list(range(6)))
        pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(run_dir / f"confusion_matrix_{test_domain}.csv")
    pd.DataFrame(test_metrics).to_csv(run_dir / "test_metrics.csv", index=False)
    (run_dir / "metadata.json").write_text(json.dumps({k: v for k,v in metadata.items() if k != "state_dict"}, indent=2))
    return test_metrics

all_test_scores = []
for mode in RUN_MODES:
    all_test_scores.extend(train_mode(mode))
if all_test_scores:
    pd.DataFrame(all_test_scores).to_csv(OUTPUT_ROOT / "all_test_metrics.csv", index=False)
    display(pd.DataFrame(all_test_scores)) if "display" in globals() else print(pd.DataFrame(all_test_scores).to_string(index=False))
print("Best checkpoints are ready for independent evaluation in", OUTPUT_ROOT)

## 7. Package outputs and verify ZIP file
Kaggle: use the **Output** tab to download `resnet50_training_results.zip`, or save each `.pth` checkpoint as a versioned Kaggle Dataset.

In [ ]:
if PACKAGE_RESULTS_AS_ZIP:
    package = OUTPUT_ROOT.parent / "resnet50_training_results.zip"
    with zipfile.ZipFile(package, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as z:
        for file in sorted(OUTPUT_ROOT.rglob("*")):
            if not file.is_file():
                continue
            # Exclude extracted input images; checkpoint+reports only.
            if any(part.startswith("_extracted_") for part in file.relative_to(OUTPUT_ROOT).parts):
                continue
            z.write(file, arcname=file.relative_to(OUTPUT_ROOT).as_posix())
    with zipfile.ZipFile(package) as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(f"Results ZIP failed CRC check: {bad}")
        checkpoint_names = [n for n in z.namelist() if n.endswith("_best.pth") and not n.endswith("_head_only_best.pth")]
        if len(checkpoint_names) != len(RUN_MODES):
            raise RuntimeError(f"Results ZIP has {len(checkpoint_names)} main checkpoints; expected {len(RUN_MODES)}")
    print(f"Verified results ZIP: {package} ({package.stat().st_size/1024**2:.1f} MiB)")
print("DONE. Test metrics were calculated using reloaded BEST-VALIDATION checkpoints.")

In [ ]:
# Persist a finish log and hashes only after previous training and ZIP cells succeed.
from pathlib import Path
_ROOT=Path(OUTPUT_ROOT)
_ARTIFACTS=list(_ROOT.rglob('*.csv'))+list(_ROOT.rglob('*.json'))+list(_ROOT.rglob('*.pth'))
append_run_log(_ROOT,kind='legacy_ResNet50_training_finished',
    config={'run_modes':RUN_MODES,'seed':SEED,'finished_after_zip':True},
    artifacts=_ARTIFACTS,repo_root=DERMSYNREAL_REPO)
print('Saved final legacy ResNet50 provenance:',_ROOT/'run_log.jsonl')
